In [15]:
# Rotate Green Line
# David H Nguyen, PhD. | TSG Lab

# This version should print the name of the file that caused an error, so you know which file it was. 

### Instructions ###
# 1. Input files MUST be in PAIRS. 1) The original image, 2) a copy of the original image. 
#   The copy image must have the characters " copy" at the end of its file name.
# 2. The copy image must have a bright green line in it. The original image cannot have green color in it. 
# 3. The rotated images will appear in the same directory that contains the input images. 
#    Rotated images will have the suffix "rotated" at the end of their file name.


In [4]:
# Load the dependencies
import os
import re
import cv2
import numpy as np

In [7]:
# Step 1 - Set Working Directory 
os.chdir('/Users/davidnguyen/Downloads/Practice Input Files')
path = os.getcwd()
contents = os.listdir(path)
hidden_files = []

# Remove hidden files (Mac-related)
for item in contents:
    if item.startswith("."):
        hidden_files.append(item)

for item in hidden_files:
    contents.remove(item)

contents

['practice-1 - Copy.png',
 'practice-4 copy.png',
 'practice-3 copy.png',
 'practice-4.png',
 'practice-3.png',
 'practice-2.png',
 'practice-2 - Copy.png',
 'practice-1.png']

In [8]:
for file_name in contents:

    match = re.search(
        r'(?:\s+-\s+copy|\s+copy)\.(png|jpg|jpeg|tif|tiff)$',
        file_name,
        flags=re.IGNORECASE
    )

    if match:

        try:
            # Detect green lines
            img = cv2.imread(os.path.join(path, file_name))
            if img is None:
                raise ValueError(f"Error loading image: {file_name}")

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            lower_green = np.array([30, 40, 40])
            upper_green = np.array([90, 255, 255])

            mask = cv2.inRange(hsv, lower_green, upper_green)

            lines = cv2.HoughLinesP(
                mask,
                1,
                np.pi / 180,
                threshold=50,
                minLineLength=10,
                maxLineGap=10
            )

            if lines is None:
                raise ValueError(f"No green lines detected in file: {file_name}")

            x1, y1, x2, y2 = lines[0][0]

            cv2.line(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

            angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))

            # Get file extension
            ext = os.path.splitext(file_name)[1]

            # Convert:
            #   practice-4 copy.png    -> practice-4.png
            #   practice-1 - Copy.png  -> practice-1.png
            original_name = re.sub(
                r'(?:\s+-\s+copy|\s+copy)(\.[^.]+)$',
                r'\1',
                file_name,
                flags=re.IGNORECASE
            )

            original_img_path = os.path.join(path, original_name)

            original_img = cv2.imread(original_img_path)
            if original_img is None:
                raise ValueError(
                    f"Error loading original image: {original_img_path}"
                )

            M = cv2.getRotationMatrix2D(
                (original_img.shape[1] // 2,
                 original_img.shape[0] // 2),
                angle,
                1
            )

            rotated_img = cv2.warpAffine(
                original_img,
                M,
                (original_img.shape[1], original_img.shape[0]),
                borderMode=cv2.BORDER_CONSTANT,
                borderValue=(255, 255, 255)
            )

            output_path = original_img_path.replace(
                ext,
                "_rotated" + ext
            )

            cv2.imwrite(output_path, rotated_img)

        except Exception as e:
            print(f"Error processing file {file_name}: {e}")